# Olist Brazilian E-Commerce Business Analysis
## Phase 4: Data Cleaning Pipeline

This notebook implements the data cleaning pipeline using modular functions written in `src/cleaning.py`. We will clean datatypes, handle missing values, resolve duplicate geolocation mappings, translate category labels, and save the output tables to `data/processed/`.

In [2]:
import os
import sys
import pandas as pd
import numpy as np

# Add the src directory to python path to import our modules
sys.path.append(os.path.abspath("../"))
from src.cleaning import (
    clean_customers,
    clean_sellers,
    clean_geolocation,
    clean_orders,
    clean_order_items,
    clean_order_payments,
    clean_order_reviews,
    clean_products
)

RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

### 1. Load Raw Datasets
Let's load the data from `data/raw/` in preparation for processing.

In [3]:
customers_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_customers_dataset.csv"))
geolocation_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_geolocation_dataset.csv"))
order_items_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_order_items_dataset.csv"))
order_payments_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_order_payments_dataset.csv"))
order_reviews_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_order_reviews_dataset.csv"))
orders_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_orders_dataset.csv"))
products_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_products_dataset.csv"))
sellers_raw = pd.read_csv(os.path.join(RAW_DIR, "olist_sellers_dataset.csv"))
category_translation = pd.read_csv(os.path.join(RAW_DIR, "product_category_name_translation.csv"))

### 2. Clean Datasets one-by-one

#### 2.1 Customers

In [4]:
customers_clean = clean_customers(customers_raw)
print(f"Customers raw shape: {customers_raw.shape} | Clean shape: {customers_clean.shape}")

Customers raw shape: (99441, 5) | Clean shape: (99441, 5)


#### 2.2 Sellers

In [5]:
sellers_clean = clean_sellers(sellers_raw)
print(f"Sellers raw shape: {sellers_raw.shape} | Clean shape: {sellers_clean.shape}")

Sellers raw shape: (3095, 4) | Clean shape: (3095, 4)


#### 2.3 Geolocation
**Note:** The original geolocation dataset contains over 1,000,000 coordinates mapping to duplicate zip code prefixes. In order to avoid a huge multi-million row cartesian explosion when joining with customers and sellers, we de-duplicate by taking the coordinate centroid (mean lat/lng) and resolving city/state modes.

In [6]:
geolocation_clean = clean_geolocation(geolocation_raw)
print(f"Geolocation raw shape: {geolocation_raw.shape} | Clean (Grouped) shape: {geolocation_clean.shape}")

Geolocation raw shape: (1000163, 5) | Clean (Grouped) shape: (19010, 5)


#### 2.4 Orders

In [7]:
orders_clean = clean_orders(orders_raw)
print(f"Orders raw shape: {orders_raw.shape} | Clean shape: {orders_clean.shape}")
print("Checking datatypes:")
print(orders_clean.dtypes)

Orders raw shape: (99441, 8) | Clean shape: (99441, 8)
Checking datatypes:
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


#### 2.5 Order Items

In [8]:
order_items_clean = clean_order_items(order_items_raw)
print(f"Order Items raw shape: {order_items_raw.shape} | Clean shape: {order_items_clean.shape}")

Order Items raw shape: (112650, 7) | Clean shape: (112650, 7)


#### 2.6 Order Payments

In [9]:
order_payments_clean = clean_order_payments(order_payments_raw)
print(f"Order Payments raw shape: {order_payments_raw.shape} | Clean shape: {order_payments_clean.shape}")

Order Payments raw shape: (103886, 5) | Clean shape: (103886, 5)


#### 2.7 Order Reviews

In [10]:
order_reviews_clean = clean_order_reviews(order_reviews_raw)
print(f"Order Reviews raw shape: {order_reviews_raw.shape} | Clean shape: {order_reviews_clean.shape}")
print(f"Null review messages left: {order_reviews_clean['review_comment_message'].isnull().sum()}")

Order Reviews raw shape: (99224, 7) | Clean shape: (99224, 7)
Null review messages left: 0


#### 2.8 Products
This joins products with the translation dictionary. Unmapped or null categories are categorized as `'unknown'`.

In [11]:
products_clean = clean_products(products_raw, category_translation)
print(f"Products raw shape: {products_raw.shape} | Clean shape: {products_clean.shape}")
print("Translated categories summary (English):")
print(products_clean['product_category_name_english'].value_counts().head(5))

Products raw shape: (32951, 9) | Clean shape: (32951, 10)
Translated categories summary (English):
product_category_name_english
bed_bath_table     3029
sports_leisure     2867
furniture_decor    2657
health_beauty      2444
housewares         2335
Name: count, dtype: int64


### 3. Verification & Checks
Let's confirm that primary key constraints are maintained, dates are parsed, and coordinate anomalies are removed.

In [12]:
assert orders_clean['order_id'].is_unique, "Order ID is not unique!"
assert customers_clean['customer_id'].is_unique, "Customer ID is not unique!"
assert products_clean['product_id'].is_unique, "Product ID is not unique!"
assert sellers_clean['seller_id'].is_unique, "Seller ID is not unique!"
assert geolocation_clean['geolocation_zip_code_prefix'].is_unique, "Geolocation Zip Code Prefix is not unique!"
print("Integrity check passed: Primary keys are unique.")

Integrity check passed: Primary keys are unique.


### 4. Export Clean Datasets
Now we save the clean dataframes to the `data/processed/` folder for use in feature engineering and EDA phases.

In [13]:
customers_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_customers_dataset.csv"), index=False)
geolocation_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_geolocation_dataset.csv"), index=False)
order_items_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_order_items_dataset.csv"), index=False)
order_payments_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_order_payments_dataset.csv"), index=False)
order_reviews_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_order_reviews_dataset.csv"), index=False)
orders_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_orders_dataset.csv"), index=False)
products_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_products_dataset.csv"), index=False)
sellers_clean.to_csv(os.path.join(PROCESSED_DIR, "olist_sellers_dataset.csv"), index=False)
print("All cleaned files successfully exported!")

All cleaned files successfully exported!
